# Crucible Quickstart

Score 3 episodes from `lerobot/aloha_static_cups_open` against any
OpenAI-compatible Qwen3-VL endpoint. Inspect the per-critic verdicts and
the aggregator's final KEEP / POLISH / REJECT.

Total runtime: ~2 minutes against a fast hosted API. ~$0.02 in API spend.

## Step 1 — Confirm the environment

In [ ]:
import os, sys
sys.path.insert(0, '..')  # so `from src import ...` works from examples/

for var in ('CRUCIBLE_VLM_ENDPOINT', 'CRUCIBLE_VLM_MODEL', 'CRUCIBLE_VLM_API_KEY'):
    val = os.environ.get(var, '<not set>')
    masked = (val[:6] + '***') if 'API_KEY' in var and val != '<not set>' else val
    print(f'  {var:30s} = {masked}')

assert os.environ.get('CRUCIBLE_VLM_ENDPOINT'), 'set CRUCIBLE_VLM_ENDPOINT before running'
assert os.environ.get('CRUCIBLE_VLM_MODEL'), 'set CRUCIBLE_VLM_MODEL before running'

## Step 2 — Pull one episode (no LLM call yet)

This validates the LeRobot v3 reader against a real Hub dataset and
shows you what the `EpisodeBundle` looks like before it goes to a critic.

In [ ]:
from src.lerobot_io import stream_episodes

bundles = list(stream_episodes(
    'lerobot/aloha_static_cups_open',
    n=1,
    frames_per_episode=8,
))

b = bundles[0]
print(f'Episode: {b.episode_index}')
print(f'Task:    {b.task_description!r}')
print(f'Camera:  {b.primary_camera}')
print(f'Frames:  {len(b.sampled_frames)} sampled at {b.sample_timestamps}')
print(f'Joints:  {b.raw_joint_states.shape if b.raw_joint_states is not None else None}')
print()
print('Telemetry digest:')
for line in b.telemetry_digest.split('\n'):
    print(' ', line)

## Step 3 — Display one of the sampled frames

In [ ]:
b.sampled_frames[0]

## Step 4 — Run all five critics + aggregator on this episode

This is the first call that actually hits your VLM endpoint. Expect
5–15 seconds total against a fast hosted API.

In [ ]:
import asyncio, json
from openai import AsyncOpenAI
from src.config import CrucibleConfig
from src.critics import run_all_critics
from src.aggregator import aggregate

cfg = CrucibleConfig()

async def score_one():
    client = AsyncOpenAI(base_url=cfg.vlm_endpoint, api_key=cfg.vlm_api_key)
    critics = await run_all_critics(b, cfg, client)
    verdict = await aggregate(critics, cfg, client)
    return critics, verdict

critics, verdict = asyncio.run(score_one())

print('=== Aggregator verdict ===')
print(json.dumps(verdict, indent=2, default=str))
print()
print('=== Per-critic outputs ===')
for name, c in critics.items():
    print(f'\n[{name}] score={c.get("score")} verdict={c.get("verdict")}')
    print(f'  rationale: {c.get("rationale")[:200]}')
    for ev in (c.get('evidence') or [])[:3]:
        print(f'  evidence:  {ev.get("timestamp")}  {ev.get("observation")}')

## Step 5 — Score 3 episodes via the full pipeline

`score_dataset` is the production path: handles caching, timeouts,
and progress callbacks. Use it (not the raw `run_all_critics` from
step 4) for any non-trivial workload.

In [ ]:
from src.pipeline import score_dataset

cfg.max_episodes_per_run = 3

results = asyncio.run(score_dataset(
    'lerobot/aloha_static_cups_open',
    cfg,
    use_cache=False,
))

for r in results:
    v = r['verdict']
    print(f"ep {r['episode_index']:3d}  score={v['final_score']:.2f}  {v['verdict']:6s}  {(v.get('top_concern') or v['summary'])[:80]}")

## Next

- [02_full_pipeline.ipynb](02_full_pipeline.ipynb) — score 25 episodes,
  histogram, threshold, push to Hub.
- [03_custom_critic.ipynb](03_custom_critic.ipynb) — add a 6th critic
  with prompt + JSON schema.